In [ ]:
!pip install mediapipe gradio
import mediapipe as mp
import cv2
import numpy as np
import gradio as gr

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [ ]:
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
def check_deadlift_feedback(landmarks):
    # Left Side Coordinates
    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]
    left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]

    # Calculate Knee Angle (Hip-Knee-Ankle) - Target: 100-140 degrees (slight bend)
    left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

    # Calculate Torso/Hip Angle (Shoulder-Hip-Knee) - Target: 120-140 degrees (flat back, hinged hip)
    left_torso_hip_angle = calculate_angle(left_shoulder, left_hip, left_knee)

    # Use left side angles for main check (assuming symmetry)
    avg_knee_angle = left_knee_angle
    avg_torso_hip_angle = left_torso_hip_angle

    # --- Feedback Logic ---
    feedback_knee = ""
    feedback_hip = ""

    # 1. Check Knee Angle (Depth)
    if avg_knee_angle < 100:
        feedback_knee = "Hips Too Low! You are Squatting"
    elif 100 <= avg_knee_angle <= 150:
        feedback_knee = "Good Knee Bend"
    else:
        feedback_knee = "Knees Too Straight! Hips Too High"

    # 2. Check Torso/Hip Angle (Back Line)
    # Target range reflects a good hip hinge (torso slightly above parallel)
    if avg_torso_hip_angle < 120:
        feedback_hip = "Torso Too Vertical! Back may be rounding"
    elif 120 <= avg_torso_hip_angle <= 160:
        feedback_hip = "Good Torso Angle"
    else:
        feedback_hip = "Torso Too Flat/Upright! Hips Too High"

    # Combine Feedback
    if "Good" in feedback_knee and "Good" in feedback_hip:
        overall_feedback = "Perfect Deadlift Setup!"
    else:
        overall_feedback = f"Knee: {feedback_knee} | Hip/Torso: {feedback_hip}"


    # --- Accuracy Calculation (Knee Target 125, Hip Target 140) ---
    knee_score = max(0, 100 - abs(avg_knee_angle - 125) * 1.5)
    hip_score = max(0, 100 - abs(avg_torso_hip_angle - 140) * 1.5)

    accuracy = (knee_score * 0.4) + (hip_score * 0.6)
    accuracy = max(0, min(100, int(accuracy)))

    return overall_feedback, int(accuracy)

In [ ]:
def analyze_deadlift(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "Error: Could not open input video."

    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    # Fix for video speed (stable FPS)
    fps = 30

    # Robust Codec Fix: Use 'MJPG' and .avi extension
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    output_video = "output_deadlift.avi"
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

    if not out.isOpened():
        return "Error: Could not create output video writer."

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Fix for mirrored video (un-mirroring webcam view)
        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Check visibility of a critical landmark before calculation
            if landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].visibility > 0.6:
                mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

                feedback, accuracy = check_deadlift_feedback(landmarks)

                color = (0, 255, 0) if "Perfect" in feedback else (0, 0, 255)

                # Using two lines to display detailed feedback
                # First line for the primary feedback
                cv2.putText(image, feedback.split('|')[0].strip(), (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)
                # Second line for the secondary feedback and accuracy
                cv2.putText(image, f"{feedback.split('|')[-1].strip()} | Accuracy: {accuracy}%", (50, 100), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)

            else:
                 cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

        else:
             cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)


        out.write(image)

    cap.release()
    out.release()
    return output_video

# Gradio Interface
gr.Interface(
    fn=analyze_deadlift,
    inputs=gr.Video(sources=["upload", "webcam"]),
    outputs=gr.Video(),
    title="Deadlift Form Analyzer",
    description="Upload a video of your deadlift, and get feedback on your setup angles!",
).launch(share=True, debug=True)